# Quantum Ising Model in 3D: Complete Tutorial

This notebook provides an interactive tutorial for simulating the **3D transverse field Ising model**.

## Hamiltonian

$$\hat{H} = -J\sum_{\langle i,j\rangle} \hat{\sigma}_i^z \hat{\sigma}_j^z - \Gamma \sum_i \hat{\sigma}_i^x$$

where:
- $J > 0$: Ferromagnetic coupling
- $\Gamma$: Transverse field strength
- $\sigma^{z,x}$: Pauli matrices

## Contents
1. [Setup and Imports](#setup)
2. [Ground State Properties](#ground-state)
3. [Quantum Phase Transition](#phase-transition)
4. [Quantum Annealing](#annealing)
5. [3D Lattice Analysis](#3d-lattice)

## 1. Setup and Imports <a name="setup"></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quantum_ising_3d import QuantumIsing3D, QuantumAnnealing, PhaseTransitionAnalyzer, visualize_results

# Configure plotting
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("✓ Imports successful!")

## 2. Ground State Properties <a name="ground-state"></a>

Let's find the ground state of a small system and compute observables.

In [ ]:
# Create 2x2x1 lattice (4 spins)
Lx, Ly, Lz = 2, 2, 1
J = 1.0
Gamma = 0.3  # Γ < J: ferromagnetic regime

model = QuantumIsing3D(Lx, Ly, Lz, J=J, Gamma=Gamma, periodic=False)

print(f"System: {Lx}×{Ly}×{Lz} lattice")
print(f"Number of spins: {model.N}")
print(f"Hilbert space dimension: {model.dim}")
print(f"Parameters: J={J}, Γ={Gamma}, Γ/J={Gamma/J:.2f}")

In [ ]:
# Find ground state and first excited state
eigenvalues, eigenvectors = model.find_ground_state(k=2)

ground_state = eigenvectors[:, 0]
E0 = eigenvalues[0]
E1 = eigenvalues[1]
gap = E1 - E0

print(f"\nEnergy spectrum:")
print(f"  E₀ = {E0:.6f}")
print(f"  E₁ = {E1:.6f}")
print(f"  Gap: ΔE = {gap:.6f}")

In [ ]:
# Compute observables
mag_z = model.magnetization_z(ground_state)
mag_x = model.magnetization_x(ground_state)
energy = model.energy(ground_state)

print(f"\nGround state observables:")
print(f"  M_z = {mag_z:.6f}")
print(f"  M_x = {mag_x:.6f}")
print(f"  Energy per spin: {energy/model.N:.6f}")

# Interpret
print(f"\nInterpretation:")
if abs(mag_z) > 0.8:
    print("  → Strong ferromagnetic order")
elif abs(mag_x) > 0.8:
    print("  → Paramagnetic phase (quantum fluctuations dominate)")
else:
    print("  → Near critical point")

### Exercise 1
Try changing `Gamma` to different values:
- Γ = 0.1 (deep in ferromagnetic phase)
- Γ = 1.0 (near critical point)
- Γ = 5.0 (paramagnetic phase)

Observe how magnetizations change!

## 3. Quantum Phase Transition <a name="phase-transition"></a>

Scan the transverse field to observe the quantum phase transition.

In [ ]:
# Use 1D chain for faster computation
Lx, Ly, Lz = 4, 1, 1
J = 1.0

analyzer = PhaseTransitionAnalyzer(Lx, Ly, Lz, J=J, periodic=True)

# Scan Γ from 0.1 to 3.0
gamma_values = np.linspace(0.1, 3.0, 12)

results = analyzer.scan_transverse_field(gamma_values, compute_gap=True)

In [ ]:
# Visualize phase transition
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Quantum Phase Transition', fontsize=16, fontweight='bold')

# Magnetizations
ax = axes[0, 0]
ax.plot(results['gamma'], np.abs(results['mag_z']), 'bo-', linewidth=2, markersize=8, label='|M_z|')
ax.plot(results['gamma'], np.abs(results['mag_x']), 'rs-', linewidth=2, markersize=8, label='|M_x|')
ax.axvline(x=1.0, color='k', linestyle='--', alpha=0.5, label='Γ_c = J (exact)')
ax.set_xlabel('Γ/J', fontsize=13)
ax.set_ylabel('Magnetization', fontsize=13)
ax.set_title('Order Parameters', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Energy
ax = axes[0, 1]
ax.plot(results['gamma'], results['energy'] / (Lx*Ly*Lz), 'go-', linewidth=2, markersize=8)
ax.axvline(x=1.0, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Γ/J', fontsize=13)
ax.set_ylabel('E₀/N', fontsize=13)
ax.set_title('Ground State Energy per Spin', fontsize=14)
ax.grid(True, alpha=0.3)

# Gap
ax = axes[1, 0]
ax.semilogy(results['gamma'], results['gap'], 'mo-', linewidth=2, markersize=8)
ax.axvline(x=1.0, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Γ/J', fontsize=13)
ax.set_ylabel('Energy Gap ΔE', fontsize=13)
ax.set_title('Excitation Gap (log scale)', fontsize=14)
ax.grid(True, alpha=0.3)

# Susceptibility (derivative of magnetization)
ax = axes[1, 1]
dmz_dgamma = np.abs(np.gradient(results['mag_z'], results['gamma']))
ax.plot(results['gamma'], dmz_dgamma, 'co-', linewidth=2, markersize=8)
ax.axvline(x=1.0, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Γ/J', fontsize=13)
ax.set_ylabel('|dM_z/dΓ|', fontsize=13)
ax.set_title('Magnetic Susceptibility', fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find critical point
critical_idx = np.argmin(results['gap'])
print(f"\nCritical point (gap minimum):")
print(f"  Γ_c/J ≈ {results['gamma'][critical_idx]:.3f}")
print(f"  Exact (1D): Γ_c/J = 1.000")

### Observations

- **Γ < Γ_c**: Ferromagnetic phase with |M_z| ≈ 1
- **Γ > Γ_c**: Paramagnetic phase with |M_x| ≈ 1
- **Γ = Γ_c**: Critical point where gap is minimum
- **Susceptibility**: Peaks near the critical point

## 4. Quantum Annealing <a name="annealing"></a>

Demonstrate quantum annealing to find ground state.

In [ ]:
# Setup annealing
Lx, Ly, Lz = 3, 1, 1
J = 1.0

model = QuantumIsing3D(Lx, Ly, Lz, J=J, periodic=False)
annealer = QuantumAnnealing(model)

# Run annealing
results = annealer.anneal(
    T=10.0,           # Total time
    num_steps=40,     # Time steps
    Gamma_init=5.0,   # Start with large Γ
    Gamma_final=0.1,  # End with small Γ
    schedule='linear' # Linear schedule
)

In [ ]:
# Visualize annealing dynamics
fig, axes = plt.subplots(3, 1, figsize=(12, 12))
fig.suptitle('Quantum Annealing Dynamics', fontsize=16, fontweight='bold')

# Transverse field schedule
ax = axes[0]
ax.plot(results['times'], results['gammas'], 'b-', linewidth=2.5)
ax.fill_between(results['times'], 0, results['gammas'], alpha=0.3)
ax.set_xlabel('Time', fontsize=13)
ax.set_ylabel('Γ(t)', fontsize=13)
ax.set_title('Transverse Field Schedule', fontsize=14)
ax.grid(True, alpha=0.3)

# Energy evolution
ax = axes[1]
ax.plot(results['times'], results['energies'], 'r-', linewidth=2.5)
ax.set_xlabel('Time', fontsize=13)
ax.set_ylabel('Energy', fontsize=13)
ax.set_title('Energy Evolution', fontsize=14)
ax.grid(True, alpha=0.3)

# Magnetizations
ax = axes[2]
ax.plot(results['times'], results['mag_z'], 'b-', linewidth=2.5, label='M_z')
ax.plot(results['times'], results['mag_x'], 'r-', linewidth=2.5, label='M_x')
ax.set_xlabel('Time', fontsize=13)
ax.set_ylabel('Magnetization', fontsize=13)
ax.set_title('Order Parameters', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAnnealing results:")
print(f"  Initial energy: {results['energies'][0]:.6f}")
print(f"  Final energy: {results['energies'][-1]:.6f}")
print(f"  Final M_z: {results['mag_z'][-1]:.6f}")
print(f"  Final M_x: {results['mag_x'][-1]:.6f}")

### Interpretation

1. **Start**: High Γ → easy quantum ground state (M_x ≈ 1)
2. **Evolution**: Gradually decrease Γ
3. **End**: Low Γ → classical ground state (aligned spins)

The system **adiabatically** follows the instantaneous ground state!

## 5. 3D Lattice Analysis <a name="3d-lattice"></a>

Explore true 3D systems.

In [ ]:
# Create 2×2×2 cubic lattice
Lx, Ly, Lz = 2, 2, 2
J = 1.0
Gamma = 0.5

model_3d = QuantumIsing3D(Lx, Ly, Lz, J=J, Gamma=Gamma, periodic=False)

print(f"3D Cubic Lattice: {Lx}×{Ly}×{Lz}")
print(f"  Total spins: {model_3d.N}")
print(f"  Hilbert space: 2^{model_3d.N} = {model_3d.dim}")

# Count bonds
bonds = 0
for i in range(model_3d.N):
    neighbors = model_3d._get_neighbors(i)
    bonds += len([j for j in neighbors if j > i])
print(f"  Total bonds: {bonds}")

# Find ground state
eigenvalues, eigenvectors = model_3d.find_ground_state(k=1)
ground_state_3d = eigenvectors[:, 0]

print(f"\nGround state:")
print(f"  E_0 = {eigenvalues[0]:.6f}")
print(f"  M_z = {model_3d.magnetization_z(ground_state_3d):.6f}")
print(f"  M_x = {model_3d.magnetization_x(ground_state_3d):.6f}")

In [ ]:
# Compute correlation matrix
corr_matrix = np.zeros((model_3d.N, model_3d.N))

for i in range(model_3d.N):
    for j in range(model_3d.N):
        if i == j:
            corr_matrix[i, j] = 1.0
        else:
            corr_matrix[i, j] = model_3d.correlation_zz(i, j, ground_state_3d)

# Visualize
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr_matrix, cmap='RdBu', vmin=-1, vmax=1, interpolation='nearest')
ax.set_xlabel('Spin j', fontsize=13)
ax.set_ylabel('Spin i', fontsize=13)
ax.set_title('Spin-Spin Correlations <σ_i^z σ_j^z> in 2×2×2 Cube', 
             fontsize=14, fontweight='bold')

# Add text annotations
for i in range(model_3d.N):
    for j in range(model_3d.N):
        text = ax.text(j, i, f'{corr_matrix[i, j]:.2f}',
                      ha="center", va="center", color="black", fontsize=9)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Correlation', fontsize=13)

ax.set_xticks(range(model_3d.N))
ax.set_yticks(range(model_3d.N))
ax.grid(False)

plt.tight_layout()
plt.show()

print("\nCorrelation structure reveals:")
print("  - Diagonal = 1 (self-correlation)")
print("  - Positive off-diagonal → ferromagnetic order")
print("  - Stronger correlation for nearby spins")

## Summary

We've explored:

1. ✓ **Ground states**: Finding and analyzing eigenstates
2. ✓ **Phase transitions**: Scanning order parameters across Γ
3. ✓ **Quantum annealing**: Adiabatic evolution protocol
4. ✓ **3D systems**: True 3D lattices and correlations

### Key Physics

- **Competition**: Classical order (J) vs quantum fluctuations (Γ)
- **Critical point**: Γ_c where gap closes and order disappears
- **Annealing**: Use quantum effects to find classical ground state

### Next Steps

- Try different lattice geometries
- Explore periodic vs open boundaries
- Compare different annealing schedules
- Study finite-size effects

For more details, see:
- `README.md` - Full documentation
- `THEORY.md` - Theoretical background
- `examples/` - More code examples